# Topological Lie Detector - Colab Deployment

This notebook orchestrates the complete training pipeline for rumor detection using TDA + Transformers.

## Instructions:
1. Upload `colab_deployment.zip` to `/content/` in Colab
2. Run all cells sequentially
3. Results will be saved to your Google Drive

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
print("✅ Google Drive mounted successfully")

In [ ]:
# Cell 2: Extract deployment package
import zipfile
import shutil

# Clean up any previous runs
if os.path.exists('/content/project'):
    shutil.rmtree('/content/project')

# Extract the deployment package
with zipfile.ZipFile('/content/colab_deployment.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/project')

print("✅ Deployment package extracted to /content/project")

# Verify structure
!ls -la /content/project/

In [ ]:
# Cell 3: Install dependencies
%cd /content/project
!pip install -q -r requirements.txt
print("✅ All dependencies installed")

In [ ]:
# Cell 4: Verify dataset
import os

dataset_path = '/content/project/data/acl2017'
if os.path.exists(dataset_path):
    twitter15_files = len(os.listdir(f'{dataset_path}/twitter15/tree'))
    twitter16_files = len(os.listdir(f'{dataset_path}/twitter16/tree'))
    print(f"✅ Dataset verified:")
    print(f"   - Twitter15: {twitter15_files} events")
    print(f"   - Twitter16: {twitter16_files} events")
else:
    print("❌ Dataset not found! Check the zip file.")

In [ ]:
# Cell 5: Run TDA-Only Experiment
print("🚀 Starting TDA-Only Experiment...")
!python run.py --config configs/colab_tda_only.yml --data_path ./data/acl2017
print("✅ TDA-Only experiment completed")

In [ ]:
# Cell 6: Run Text-Only Experiment
print("🚀 Starting Text-Only Experiment...")
!python run.py --config configs/colab_text_only.yaml --data_path ./data/acl2017
print("✅ Text-Only experiment completed")

In [ ]:
# Cell 7: Run Hybrid Experiment
print("🚀 Starting Hybrid Experiment...")
!python run.py --config configs/colab_hybrid.yaml --data_path ./data/acl2017
print("✅ Hybrid experiment completed")

In [ ]:
# Cell 8: Package and save results to Google Drive
import zipfile
import datetime
import os

# Create timestamp for unique filename
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
results_zip = f'/content/rumor_detection_results_{timestamp}.zip'

# Create zip file with results and checkpoints
with zipfile.ZipFile(results_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add results
    for root, dirs, files in os.walk('/content/project/results'):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, '/content/project')
            zipf.write(file_path, arcname)
    
    # Add checkpoints
    for root, dirs, files in os.walk('/content/project/checkpoints'):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, '/content/project')
            zipf.write(file_path, arcname)

print(f"✅ Results packaged: {results_zip}")

# Create Drive directory if it doesn't exist
drive_results_dir = '/content/drive/MyDrive/Project_Results'
os.makedirs(drive_results_dir, exist_ok=True)

# Copy to Google Drive
import shutil
drive_destination = f'{drive_results_dir}/rumor_detection_results_{timestamp}.zip'
shutil.copy(results_zip, drive_destination)

print(f"✅ Results saved to Google Drive: {drive_destination}")
print("\n📊 Summary:")
print(f"   - Results directory: {drive_results_dir}")
print(f"   - Filename: rumor_detection_results_{timestamp}.zip")

In [ ]:
# Cell 9: Display final metrics summary
import json
import os

print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60 + "\n")

modes = ['tda_only', 'text_only', 'hybrid']
for mode in modes:
    metrics_file = f'/content/project/results/{mode}/test_metrics.json'
    if os.path.exists(metrics_file):
        with open(metrics_file, 'r') as f:
            metrics = json.load(f)
        print(f"\n{mode.upper().replace('_', ' ')}:")
        print(f"  Accuracy:  {metrics.get('accuracy', 0):.4f}")
        print(f"  F1 Score:  {metrics.get('f1', 0):.4f}")
        print(f"  Precision: {metrics.get('precision', 0):.4f}")
        print(f"  Recall:    {metrics.get('recall', 0):.4f}")
    else:
        print(f"\n{mode.upper().replace('_', ' ')}: Metrics file not found")

print("\n" + "="*60)
print("✅ All experiments completed successfully!")
print("="*60)